In [2]:
import os
import subprocess

mp3_dir = "outputs/mp3"
wav_dir = "outputs/wav"
os.makedirs(wav_dir, exist_ok=True)

base_name = input("请输入要转换的文件名（不含后缀）：").strip()

mp3_path = os.path.join(mp3_dir, base_name + ".mp3")
wav_path = os.path.join(wav_dir, base_name + ".wav")

if not os.path.exists(mp3_path):
    print("❌ 找不到该 MP3 文件")
else:
    cmd = [
        "ffmpeg",
        "-y",              # 覆盖输出
        "-loglevel", "error",  # ✅ 只显示错误，不刷屏
        "-i", mp3_path,
        "-ar", "16000",
        "-ac", "1",
        wav_path
    ]
    subprocess.run(cmd, check=True)
    print(f"✅ 转换完成：{wav_path}")

请输入要转换的文件名（不含后缀）：Mt_1_en
✅ 转换完成：outputs/wav/Mt_1_en.wav


In [9]:
import os
import subprocess

# 找到 aligner 环境路径
conda_prefix = subprocess.check_output(
    ["conda", "info", "--base"], text=True
).strip()

aligner_bin = os.path.join(conda_prefix, "envs", "aligner", "bin")
os.environ["PATH"] = aligner_bin + ":" + os.environ["PATH"]

# 验证
!mfa version

python(84858) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.
python(84859) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


3.3.9


In [10]:
import os
import shutil

os.makedirs("corpus", exist_ok=True)

base = "Mt_1_en"

shutil.copy(f"outputs/wav/{base}.wav", f"corpus/{base}.wav")
shutil.copy(f"outputs/plaintext/{base}.txt", f"corpus/{base}.txt")

print("✅ corpus 准备完成")

✅ corpus 准备完成


In [11]:
!mfa align corpus \
  english_mfa \
  english_mfa \
  forcealign \
  --clean \
  --overwrite

python(84922) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


 INFO     Setting up corpus information...                                      
 INFO     Loading corpus from source files...                                   
   1% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/100  [ 0:00:01 < -:--:-- , ? it/s ]
 INFO     Found 1 speaker across 1 file, average number of utterances per       
          speaker: 1.0                                                          
 INFO     Initializing multiprocessing jobs...                                  
 WARNING  Number of jobs was specified as 3, but due to only having 1 speakers, 
          MFA will only use 1 jobs. Use the --single_speaker flag if you would  
          like to split utterances across jobs regardless of their speaker.     
 INFO     Normalizing text...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/1  [ 0:00:01 < 0:00:00 , ? it/s ]
 INFO     Generating MFCCs...                                                   
 100% ━━━━━━━━━━━━━━━━━━━━━━

In [18]:
# 连接数据库

import sqlite3

conn = sqlite3.connect("db/bible.db")
cursor = conn.cursor()

In [21]:
# 清空数据库时间戳

cursor.execute("""
UPDATE words
SET start_time = NULL,
    end_time = NULL;
""")
conn.commit()

In [22]:
import sqlite3
import re

def parse_textgrid(textgrid_path):
    """解析TextGrid文件，提取单词的时间戳"""
    time_intervals = []
    current_interval = None
    in_words_tier = False

    with open(textgrid_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # 检测是否进入words层（IntervalTier）
            if line.startswith('name = "words"'):
                in_words_tier = True
                continue
            elif line.startswith('item ['):  # 遇到下一个层，退出words层
                in_words_tier = False
                continue

            if in_words_tier:
                # 检测区间开始（intervals [N]:）
                interval_match = re.match(r'intervals\s*\[\s*(\d+)\s*\]\s*:', line)
                if interval_match:
                    current_interval = {
                        'index': int(interval_match.group(1)),
                        'xmin': None,
                        'xmax': None,
                        'text': ''
                    }
                    continue

                # 提取xmin
                xmin_match = re.match(r'xmin\s*=\s*([\d\.]+)', line)
                if xmin_match and current_interval:
                    current_interval['xmin'] = float(xmin_match.group(1))
                    continue

                # 提取xmax
                xmax_match = re.match(r'xmax\s*=\s*([\d\.]+)', line)
                if xmax_match and current_interval:
                    current_interval['xmax'] = float(xmax_match.group(1))
                    continue

                # 提取text
                text_match = re.match(r'text\s*=\s*"([^"]*)"', line)
                if text_match and current_interval:
                    current_interval['text'] = text_match.group(1).strip()
                    # 过滤空文本或仅空格的文本
                    if current_interval['text']:
                        time_intervals.append((
                            current_interval['xmin'],
                            current_interval['xmax'],
                            current_interval['text']
                        ))
                    current_interval = None  # 重置当前区间，准备下一个

    return time_intervals

def update_words_timestamps(db_path, textgrid_path):
    """更新words表的时间戳"""
    # 1. 解析TextGrid获取单词时间戳
    time_intervals = parse_textgrid(textgrid_path)
    if not time_intervals:
        print("未从TextGrid中提取到有效单词时间戳！")
        return

    # 2. 连接数据库
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    try:
        # 3. 遍历words表，按顺序更新时间戳
        word_index = 0  # 跟踪time_intervals的索引
        cursor.execute("SELECT id, word FROM words WHERE type = 'word' ORDER BY id")
        word_records = cursor.fetchall()

        for record in word_records:
            word_id, word_text = record
            if word_index >= len(time_intervals):
                print(f"警告：TextGrid中的单词数量少于words表中的单词数量（{len(time_intervals)} < {len(word_records)}）")
                break

            # 匹配单词文本（大小写不敏感匹配）
            tg_xmin, tg_xmax, tg_word = time_intervals[word_index]
            if word_text.lower() == tg_word.lower():
                # 更新时间戳
                cursor.execute(
                    "UPDATE words SET start_time = ?, end_time = ? WHERE id = ?",
                    (tg_xmin, tg_xmax, word_id)
                )
                word_index += 1
            else:
                print(f"警告：单词不匹配！words表：{word_text}，TextGrid：{tg_word}（ID：{word_id}）")

        # 提交更改
        conn.commit()
        print(f"成功更新 {word_index} 个单词的时间戳！")

    except Exception as e:
        print(f"更新过程中出错：{e}")
        conn.rollback()
    finally:
        conn.close()

# ------------------- 调用示例 -------------------
db_path = "db/bible.db"       # 数据库路径
textgrid_path = "forcealign/Mt_1_en.TextGrid" # TextGrid文件路径
update_words_timestamps(db_path, textgrid_path)

成功更新 533 个单词的时间戳！
